<a href="https://colab.research.google.com/github/Kamaumbugua-dev/Kamaumbugua-dev/blob/main/Route_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Boda Boda Route & Maintenance Predictor
---------------------------------------
This program:
1. Generates (synthetic) ride data
2. Trains:
   - Travel time predictor
   - Maintenance predictor
3. Builds a Markov chain for route prediction
4. Provides helper functions to:
   - Suggest best route & departure time
   - Predict when maintenance is needed
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib
from collections import defaultdict

# ----------------------
# 1. Generate synthetic dataset
# ----------------------
np.random.seed(42)
n = 4000
places = ['home','market','station','airport','mall','school','hospital']

def rand_place(): return np.random.choice(places)

data = []
for i in range(n):
    origin = rand_place()
    dest = np.random.choice([p for p in places if p!= origin])
    distance_km = np.round(np.random.uniform(1, 12),2)
    time_of_day = np.random.choice(['early_morning','morning_peak','midday','evening_peak','night'])
    traffic_map = {'early_morning':0.2,'morning_peak':0.8,'midday':0.4,'evening_peak':0.9,'night':0.2}
    traffic = np.clip(np.random.normal(traffic_map[time_of_day], 0.1), 0, 1)
    road_condition = np.random.choice(['good','fair','poor'], p=[0.6,0.3,0.1])
    road_score = {'good':1.0,'fair':0.7,'poor':0.4}[road_condition]
    weather = np.random.choice(['clear','rainy','windy'], p=[0.75,0.2,0.05])
    weather_penalty = {'clear':1.0,'rainy':1.25,'windy':1.1}[weather]

    avg_speed = np.clip(np.random.normal(30 - 10*traffic, 5), 8, 70)
    harsh_brakes = np.random.poisson(0.3 + traffic*1.5)
    rapid_accels = np.random.poisson(0.2 + (1-traffic)*0.8)
    avg_load_kg = np.round(np.random.choice([0,5,10,15,20], p=[0.3,0.25,0.2,0.15,0.1]),1)

    base_time = distance_km / (avg_speed/60.0)
    travel_time = base_time * (1 + traffic*0.6) * (1/road_score) * weather_penalty
    travel_time = np.round(np.random.normal(travel_time, travel_time*0.08),1)

    wear_factor = 1 + harsh_brakes*0.03 + (1-road_score)*0.5 + (avg_load_kg/50)
    km_between_services = np.clip(np.round(np.random.normal(2000 / wear_factor, 150)), 300, 5000)
    cumulative_km = np.random.randint(0, km_between_services)
    km_to_service = km_between_services - cumulative_km + np.random.randint(-20,20)

    data.append({
        'origin':origin, 'dest':dest, 'distance_km':distance_km, 'time_of_day':time_of_day,
        'traffic':traffic, 'road_condition':road_condition, 'road_score':road_score,
        'weather':weather, 'avg_speed_kmh':avg_speed,'harsh_brakes':harsh_brakes,'rapid_accels':rapid_accels,
        'avg_load_kg':avg_load_kg, 'travel_time_min':np.round(travel_time,1),
        'km_to_service': int(km_to_service)
    })

df = pd.DataFrame(data)

# ----------------------
# 2. Train ML models
# ----------------------
df2 = pd.get_dummies(df, columns=['time_of_day','road_condition','weather','origin','dest'], drop_first=True)

# Travel time model
X_time = df2.drop(columns=['travel_time_min','km_to_service'])
y_time = df2['travel_time_min']
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_time, y_time, test_size=0.2, random_state=42)
time_model = RandomForestRegressor(n_estimators=80, random_state=42, max_depth=12)
time_model.fit(X_train_t, y_train_t)
mae_time = mean_absolute_error(y_test_t, time_model.predict(X_test_t))

# Maintenance model
X_maint = df2.drop(columns=['travel_time_min','km_to_service'])
y_maint = df2['km_to_service']
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_maint, y_maint, test_size=0.2, random_state=42)
maint_model = RandomForestRegressor(n_estimators=80, random_state=42, max_depth=12)
maint_model.fit(X_train_m, y_train_m)
mae_maint = mean_absolute_error(y_test_m, maint_model.predict(X_test_m))

print(f"Travel-time model MAE: {mae_time:.2f} minutes")
print(f"Maintenance model MAE: {mae_maint:.1f} km\n")

joblib.dump(time_model, "time_model.joblib")
joblib.dump(maint_model, "maint_model.joblib")

# ----------------------
# 3. Markov chain for route prediction
# ----------------------
sequences = []
for i in range(800):
    seq_length = np.random.randint(3,9)
    seq = list(np.random.choice(places, size=seq_length, replace=True))
    sequences.append(seq)

trans_counts = defaultdict(lambda: defaultdict(int))
for seq in sequences:
    for a,b in zip(seq, seq[1:]):
        trans_counts[a][b] += 1

trans_probs = {}
for a,targets in trans_counts.items():
    total = sum(targets.values())
    probs = {b: targets[b]/total for b in targets}
    trans_probs[a] = probs

print("Example transition probabilities from 'home':")
print(trans_probs.get('home', {}))

# ----------------------
# 4. Helper functions
# ----------------------
def prepare_input_row(origin, dest, distance_km, time_of_day, road_condition, weather,
                      avg_speed_kmh, harsh_brakes, rapid_accels, avg_load_kg):
    row = {
        'distance_km': distance_km, 'traffic': np.clip(np.random.normal(0.5,0.2),0,1),
        'road_score': {'good':1.0,'fair':0.7,'poor':0.4}[road_condition],
        'avg_speed_kmh': avg_speed_kmh, 'harsh_brakes':harsh_brakes, 'rapid_accels':rapid_accels,
        'avg_load_kg':avg_load_kg
    }
    temp = pd.DataFrame([row])
    temp = pd.concat([temp, pd.get_dummies(pd.DataFrame([{
        'time_of_day':time_of_day,'road_condition':road_condition,'weather':weather,
        'origin':origin,'dest':dest
    }]), columns=['time_of_day','road_condition','weather','origin','dest'], drop_first=True)], axis=1)
    for c in X_time.columns:
        if c not in temp.columns:
            temp[c] = 0
    temp = temp[X_time.columns]
    return temp

def suggest_best_route_and_time(origin, dest, candidate_routes, departure_time_options, rider_metrics):
    best = None
    results = []
    for route in candidate_routes:
        for tod in departure_time_options:
            Xrow = prepare_input_row(origin, dest, route['distance_km'], tod,
                                     route['road_condition'], route.get('weather','clear'),
                                     route.get('avg_speed_kmh',25),
                                     rider_metrics.get('harsh_brakes',0),
                                     rider_metrics.get('rapid_accels',0),
                                     rider_metrics.get('avg_load_kg',0))
            pred_time = time_model.predict(Xrow)[0]
            results.append({'route':route['name'],'departure':tod,'pred_time_min':pred_time,'distance_km':route['distance_km']})
            if best is None or pred_time < best['pred_time_min']:
                best = results[-1]
    res_df = pd.DataFrame(results).sort_values('pred_time_min').reset_index(drop=True)
    return best, res_df

def predict_maintenance(origin, dest, distance_km, time_of_day, road_condition, weather,
                        rider_metrics, current_km_since_service):
    Xrow = prepare_input_row(origin, dest, distance_km, time_of_day, road_condition, weather,
                             rider_metrics.get('avg_speed_kmh',25),
                             rider_metrics.get('harsh_brakes',0),
                             rider_metrics.get('rapid_accels',0),
                             rider_metrics.get('avg_load_kg',0))
    pred_km_to_service = maint_model.predict(Xrow)[0]
    predicted_km_remaining = max(0, int(pred_km_to_service - current_km_since_service))
    return int(pred_km_to_service), predicted_km_remaining

# ----------------------
# 5. Demonstration
# ----------------------
origin = 'home'; dest = 'market'
candidate_routes = [
    {'name':'Route A (via Main Rd)','distance_km':5.6,'road_condition':'good','avg_speed_kmh':32,'weather':'clear'},
    {'name':'Route B (short-cut)','distance_km':4.7,'road_condition':'poor','avg_speed_kmh':20,'weather':'clear'},
    {'name':'Route C (long but smooth)','distance_km':7.8,'road_condition':'good','avg_speed_kmh':35,'weather':'clear'}
]
departure_time_options = ['early_morning','morning_peak','midday','evening_peak','night']
rider_metrics = {'harsh_brakes':2,'rapid_accels':1,'avg_load_kg':8,'avg_speed_kmh':28}

best, candidates_df = suggest_best_route_and_time(origin, dest, candidate_routes, departure_time_options, rider_metrics)
print("\nBest suggestion based on predicted travel time:")
print(best)

print("\nTop candidate route/time predictions:")
print(candidates_df.head(6))

# Maintenance prediction demo
current_km_since_service = 1800
pred_km_to_service, predicted_remaining = predict_maintenance(
    origin,dest,5.6,'morning_peak','good','clear',rider_metrics,current_km_since_service
)
print(f"\nMaintenance prediction: ~{pred_km_to_service} km until next service.")
print(f"Given current {current_km_since_service} km since last service, estimated remaining: {predicted_remaining} km.")

Travel-time model MAE: 2.93 minutes
Maintenance model MAE: 402.9 km

Example transition probabilities from 'home':
{np.str_('mall'): 0.15300546448087432, np.str_('hospital'): 0.15482695810564662, np.str_('market'): 0.1366120218579235, np.str_('station'): 0.14389799635701275, np.str_('airport'): 0.15664845173041894, np.str_('home'): 0.1111111111111111, np.str_('school'): 0.14389799635701275}

Best suggestion based on predicted travel time:
{'route': 'Route A (via Main Rd)', 'departure': 'morning_peak', 'pred_time_min': np.float64(15.321156148009123), 'distance_km': 5.6}

Top candidate route/time predictions:
                       route      departure  pred_time_min  distance_km
0      Route A (via Main Rd)   morning_peak      15.321156          5.6
1      Route A (via Main Rd)         midday      15.813057          5.6
2      Route A (via Main Rd)          night      16.049723          5.6
3      Route A (via Main Rd)  early_morning      18.673371          5.6
4      Route A (via Main 